In [ ]:
import os
import time
import pandas as pd
from alpha_vantage.timeseries import TimeSeries
from dotenv import load_dotenv


In [4]:
load_dotenv()
ALPHA_VANTAGE_API_KEY = os.getenv("ALPHA_VANTAGE_API_KEY")
if not ALPHA_VANTAGE_API_KEY:
    raise ValueError("ALPHA_VANTAGE_API_KEY is not set in the environment variables.")

In [5]:
tickers = [
    "AAPL",  # Apple Inc.
    "MSFT",  # Microsoft Corporation
    "NVDA",  # NVIDIA Corporation
    "AMZN",  # Amazon.com Inc.
    "GOOGL",  # Alphabet Inc. (Google)
]

ts = TimeSeries(key=ALPHA_VANTAGE_API_KEY, output_format="pandas")

def fetch_daily_stock_data(symbol, outputsize="full"):
    print(f"Fetching data for {symbol}...")
    data, _ = ts.get_daily(symbol=symbol, outputsize=outputsize)
    print(data.columns)
    data = data.rename(columns={
        "1. open": "Open",
        "2. high": "High",
        "3. low": "Low",
        "4. close": "Close",
        "5. volume": "Volume"
    })
    data = data[["Open", "High", "Low", "Close", "Volume"]]
    data.index.name = "Date"
    data = data.sort_index()  # ascending by date
    return data

In [19]:
SAVE_DIR = "../data/raw"
os.makedirs(SAVE_DIR, exist_ok=True)

for ticker in tickers:
    try:
        df = fetch_daily_stock_data(ticker)
        df = df[df.index >= "2014-01-01"]  # last 10 years
        df.to_csv(f"{SAVE_DIR}/{ticker}.csv")
        print(f"{ticker}: {len(df)} rows saved.\n")
        time.sleep(15)  # Respect API limits
    except Exception as e:
        print(f"Error fetching {ticker}: {e}")


Fetching data for AAPL...
Index(['1. open', '2. high', '3. low', '4. close', '5. volume'], dtype='object')
AAPL: 2893 rows saved.

Fetching data for MSFT...
Index(['1. open', '2. high', '3. low', '4. close', '5. volume'], dtype='object')
MSFT: 2893 rows saved.

Fetching data for NVDA...
Index(['1. open', '2. high', '3. low', '4. close', '5. volume'], dtype='object')
NVDA: 2893 rows saved.

Fetching data for AMZN...
Index(['1. open', '2. high', '3. low', '4. close', '5. volume'], dtype='object')
AMZN: 2893 rows saved.

Fetching data for GOOGL...
Index(['1. open', '2. high', '3. low', '4. close', '5. volume'], dtype='object')
GOOGL: 2893 rows saved.



In [7]:
SAVE_DIR = "../data/raw"
for ticker in tickers:
    file_path = os.path.join(SAVE_DIR, f"{ticker}.csv")
    try:
        df = pd.read_csv(file_path, parse_dates=["Date"], index_col="Date")
        print (f"\n{ticker} - HEAD OF DATAFRAME:\n", df.head())
    except Exception as e:
        print(f"Error reading {ticker} data: {e}")


AAPL - HEAD OF DATAFRAME:
               Open    High      Low     Close      Volume
Date                                                     
2014-01-02  555.68  557.03  552.021  553.1300   8381600.0
2014-01-03  552.86  553.70  540.430  540.9800  14016700.0
2014-01-06  537.45  546.80  533.600  543.9300  14736100.0
2014-01-07  544.32  545.96  537.925  540.0375  11328900.0
2014-01-08  538.81  545.56  538.690  543.4600   9233200.0

MSFT - HEAD OF DATAFRAME:
               Open   High    Low  Close      Volume
Date                                               
2014-01-02  37.350  37.40  37.10  37.16  30632200.0
2014-01-03  37.200  37.22  36.60  36.91  31134800.0
2014-01-06  36.850  36.89  36.11  36.13  43603700.0
2014-01-07  36.325  36.49  36.21  36.41  35802800.0
2014-01-08  36.000  36.14  35.58  35.76  59971700.0

NVDA - HEAD OF DATAFRAME:
              Open     High     Low  Close      Volume
Date                                                 
2014-01-02  15.92  15.9800  15.720  15

In [8]:
DATA_DIR = "../data/raw"
tickers = [
    "AAPL",  # Apple Inc.
    "MSFT",  # Microsoft Corporation
    "NVDA",  # NVIDIA Corporation
    "AMZN",  # Amazon.com Inc.
    "GOOGL",  # Alphabet Inc. (Google)
]

def load_and_tag_csv(ticker):
    path = os.path.join(DATA_DIR, f"{ticker}.csv")
    df = pd.read_csv(path, parse_dates=["Date"], index_col="Date")
    df["Ticker"] = ticker
    return df

stock_dfs = [load_and_tag_csv(ticker) for ticker in tickers]
merged_df = pd.concat(stock_dfs)
merged_df = merged_df.sort_index()  # Ensure the DataFrame is sorted by date
print(f"Combined DataFrame shape: {merged_df.shape}")
print(merged_df.head())

Combined DataFrame shape: (14465, 6)
               Open     High       Low    Close      Volume Ticker
Date                                                              
2014-01-02   555.68   557.03   552.021   553.13   8381600.0   AAPL
2014-01-02   398.80   399.36   394.020   397.97   2137800.0   AMZN
2014-01-02    37.35    37.40    37.100    37.16  30632200.0   MSFT
2014-01-02  1115.46  1117.75  1108.260  1113.12   3639100.0  GOOGL
2014-01-02    15.92    15.98    15.720    15.86   6502300.0   NVDA


In [9]:
date_sets = [set(df.index.date) for df in stock_dfs]
common_dates = sorted(set.intersection(*date_sets))

merged_df = merged_df.loc[merged_df.index.isin(common_dates)] # Ensure only common dates are kept

merged_df = merged_df.sort_index()  # Ensure the DataFrame is sorted by date

print(f"Filtered DataFrame shape: {merged_df.shape}")
print(merged_df.head())

Filtered DataFrame shape: (14465, 6)
               Open     High       Low    Close      Volume Ticker
Date                                                              
2014-01-02   555.68   557.03   552.021   553.13   8381600.0   AAPL
2014-01-02   398.80   399.36   394.020   397.97   2137800.0   AMZN
2014-01-02    37.35    37.40    37.100    37.16  30632200.0   MSFT
2014-01-02  1115.46  1117.75  1108.260  1113.12   3639100.0  GOOGL
2014-01-02    15.92    15.98    15.720    15.86   6502300.0   NVDA


/var/folders/60/w45408950l338hfzlxx878vc0000gn/T/ipykernel_76330/1274865240.py:4: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  merged_df = merged_df.loc[merged_df.index.isin(common_dates)] # Ensure only common dates are kept


In [10]:
merged_df.to_csv("../data/raw/multi_stock_merged.csv")